In [13]:
import pandas as pd
import pickle

df  = pd.read_parquet("data.parquet")

# affiche les nombres sans notation scientifique avec pandas
pd.set_option("display.float_format", "{:,.2f}".format)

X_test  = pd.read_parquet("X_test.parquet")
y_test  = pd.read_parquet("y_test.parquet")

# dans l'approche LLM, il n'est pas possible de prédire un Tag sans le texte de réclamation
filter = (X_test["Consumer Claim"].isna() == False)
X_test  = X_test[filter]
y_test  = y_test[filter]

# Chargement des catégories
with open("categories.pkl", "rb") as f:
    categories = pickle.load(f)

# stockage des metriques
global_metrics = []

# Approche LLM

On demande au LLM de choisir parmis la catégorie 'Tag' en fonction de la demande client 'Consumer Claim'

In [14]:
import os
import pandas as pd

from mistralai.client import Mistral


# Client Mistral
client = Mistral(
    api_key=os.environ["MISTRAL_API_KEY"]
)


# ---------------------------------------------------------
# Préparation du prompt système
# ---------------------------------------------------------

SYSTEM_PROMPT = """
Vous êtes un système de classification de réclamations clientes.

Votre tâche consiste à attribuer à chaque réclamation UNE SEULE catégorie parmi les catégories autorisées.

Vous devez retourner exactement le nom d'une catégorie présente dans la liste fournie, sans explication supplémentaire.

Catégories autorisées :
{categories}

{additional_prompt}
"""


# ---------------------------------------------------------
# Fonction de classification
# ---------------------------------------------------------

def classify_with_llm(
    claim: str,
    categories: list[str],
    model: str = os.environ["MISTRAL_MODEL"],
    temperature: float = 0.0,
    additional_prompt: str = None
) -> str:
    """
    Classifie une réclamation client à l'aide d'un modèle de langage Mistral.

    La fonction construit un prompt système contenant la liste des catégories
    autorisées, puis soumet la réclamation au modèle afin qu'il détermine la
    catégorie correspondante.

    Args:
        claim (str):
            Texte de la réclamation client à classifier.

        categories (list[str]):
            Liste des catégories autorisées pour la classification.
            Le modèle doit retourner exactement l'une de ces catégories.

        model (str, optional):
            Identifiant du modèle Mistral utilisé pour la classification.
            Par défaut, "mistral-small-latest".

        temperature (float, optional):
            Paramètre contrôlant le niveau de variabilité de la réponse.
            Une valeur de 0.0 est utilisée par défaut afin de rendre la
            classification aussi déterministe que possible.

    Returns:
        str:
            Catégorie prédite par le modèle. Les espaces superflus au début
            et à la fin de la réponse sont supprimés.

    Raises:
        Exception:
            Une exception peut être levée si l'appel à l'API Mistral échoue
            ou si la réponse retournée ne possède pas le format attendu.

    Example:
        categories = [
            "Checking or savings account",
            "Debt collection",
            "Mortgage",
        ]

        category = classify_with_llm(
            claim="I have a problem with my mortgage payment.",
            categories=categories,
        )

        print(category)
    """

    #print(model)

    system_prompt = SYSTEM_PROMPT.format(
        categories="\n".join(f"- {category}" for category in categories),
        additional_prompt=additional_prompt
    )

    #print(system_prompt)

    user_prompt = f"""
Réclamation à classifier :

{claim}

Retournez uniquement la catégorie correspondante.
"""

    #print(user_prompt)

    response = client.chat.complete(
        model=model,
        temperature=temperature,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
    )

    return response.choices[0].message.content.strip(), response.usage

In [15]:
import random
import time

def with_retry(func: callable, max_retries=5):
    """
    Exécute une fonction avec une stratégie de nouvelle tentative automatique.

    La fonction est exécutée et, en cas d'exception, elle est relancée
    automatiquement jusqu'à atteindre le nombre maximal de tentatives.
    Le délai entre les tentatives augmente de manière exponentielle et
    une durée aléatoire est ajoutée afin d'éviter que plusieurs appels
    simultanés ne soient effectués exactement au même moment.

    Lorsque le nombre maximal de tentatives est atteint, l'utilisateur
    peut choisir de poursuivre le traitement en réinitialisant le compteur
    de tentatives ou d'arrêter le traitement.

    Args:
        func (callable):
            Fonction sans argument à exécuter.

        max_retries (int, optional):
            Nombre maximal de tentatives consécutives avant de demander
            à l'utilisateur s'il souhaite poursuivre. Par défaut à 5.

    Returns:
        Any:
            Résultat retourné par `func` lorsque son exécution réussit.

        None:
            Si le nombre maximal de tentatives est atteint et que
            l'utilisateur choisit d'arrêter le traitement.

    Raises:
        Exception:
            Les exceptions levées par `func` sont interceptées. Elles ne
            sont donc pas propagées lorsque l'utilisateur choisit de
            poursuivre ou d'arrêter le traitement.

    Notes:
        Le délai avant chaque nouvelle tentative suit la formule :

            2 ** attempt + random.uniform(0, 1)

        Il augmente donc progressivement afin de limiter les appels
        répétés et rapprochés vers le service distant.

    Example:
        result = with_retry(
            lambda: client.chat.complete(
                model="mistral-small-latest",
                messages=messages
            )
        )
    """
    attempt = 0

    while True:
        try:
            return func()

        except Exception as e:
            attempt += 1

            if attempt >= max_retries:
                print(f"Échec de l'appel : {e}")

                answer = input(
                    "Le nombre de tentatives est écoulé. "
                    "Continuer tout de même ? (o/n) : "
                ).strip().lower()

                if answer not in ["o", "oui", "y", "yes"]:
                    print("Traitement arrêté.")
                    return None

                # Nouvelle série de tentatives
                attempt = 0
                continue

            wait_time = (
                2 ** attempt
                + random.uniform(0, 1)
            )

            print(f"Erreur de l'appel : {e}")
            print(
                f"Nouvelle tentative dans "
                f"{wait_time:.2f} secondes..."
            )

            time.sleep(wait_time)

# Fonction affiche le coût estimé des tokens / €

In [16]:
def print_tokens_cost(model, results):
    input_price = float(os.environ["MISTRAL_MODEL_INPUT_COST_1M_TOKEN"] if model == os.environ["MISTRAL_MODEL"] else os.environ["MISTRAL_LARGE_MODEL_INPUT_COST_1M_TOKEN"])
    outut_price = float(os.environ["MISTRAL_MODEL_OUTPUT_COST_1M_TOKEN"] if model == os.environ["MISTRAL_MODEL"] else os.environ["MISTRAL_LARGE_MODEL_OUTPUT_COST_1M_TOKEN"])

    input_cost = (results["Input Tokens"].sum() * input_price)  / 1_000_000.0
    output_cost = (results["Output Tokens"].sum() * outut_price)  / 1_000_000.0
    total_cost = input_cost + output_cost
    
    print(
        "Tokens:\n",
        "Coût entrée ($)", f"{input_cost:.6f}", "\n",
        "Coût sortie ($)", f"{output_cost:.6f}", "\n",
        "Coût total  ($)", f"{total_cost:.6f}", "\n",
    )

# Fonctions de stockage des résultats

In [17]:
import pickle
from pathlib import Path

def load_results(results_path):
    print("load_results", results_path)
    if results_path.exists():
        # sauvegarde également les différentes catègories
        with open(results_path, "rb") as f:
            return pickle.load(f)
    else:
        return pd.DataFrame()
        
def save_results(results, results_path):
    # sauvegarde également les différentes catègories
    with open(results_path, "wb") as f:
        pickle.dump(results, f)

# Fonction de test

In [18]:
from collections.abc import Callable
import time
import pandas as pd


def test(
    indices,
    additional_prompt: str | Callable[[str], str] | None = None,
    **kwargs
) -> pd.DataFrame:
    """
    Évalue les performances du modèle de classification sur un ensemble
    d'indices du jeu de test.

    Pour chaque réclamation, la fonction construit éventuellement un prompt
    complémentaire, puis effectue la classification via `with_retry`.
    Le prompt complémentaire peut être fourni directement sous forme de
    chaîne de caractères ou être généré dynamiquement par une fonction.

    Args:
        indices:
            Collection d'indices correspondant aux lignes de `X_test` et
            `y_test` à utiliser pour l'évaluation.

        additional_prompt (str | Callable[[str], str] | None, optional):
            Prompt complémentaire à ajouter au prompt de classification.

            Trois comportements sont possibles :

            - `None` : aucun prompt complémentaire.
            - `str` : le même prompt est utilisé pour toutes les questions.
            - `Callable[[str], str]` : la fonction est appelée pour chaque
              question avec le texte de la réclamation comme argument.
              Elle doit retourner le prompt complémentaire à utiliser.

    Returns:
        pd.DataFrame:
            Tableau récapitulatif contenant les résultats de chaque test.

            Colonnes :
            - `Index` : index de la réclamation.
            - `Question` : réclamation testée.
            - `Réponse` : catégorie prédite.
            - `Attendue` : catégorie réelle.
            - `Correct` : indique si la prédiction est correcte.
            - `Temps (s)` : temps nécessaire pour obtenir la réponse.

    Raises:
        TypeError:
            Si `additional_prompt` n'est ni `None`, ni une chaîne de
            caractères, ni une fonction appelable.

    Example:
        # Prompt fixe
        results = test(
            indices,
            additional_prompt="Soyez particulièrement attentif..."
        )

        # Prompt généré dynamiquement
        def get_examples(question):
            examples = retrieve_examples(
                metadata,
                question,
                k=3
            )

            return format_examples(examples)

        results = test(
            indices,
            additional_prompt=get_examples
        )
    """

    results = []

    for idx in indices:
        X = X_test.loc[idx]
        y = y_test.loc[idx]

        question = X["Consumer Claim"]
        expected = y["Tag"]

        # Génération du prompt complémentaire
        if additional_prompt is None:
            prompt = None
        elif isinstance(additional_prompt, str):
            prompt = additional_prompt
        elif callable(additional_prompt):
            prompt = additional_prompt(question)
        else:
            raise TypeError(
                "additional_prompt doit être None, "
                "une chaîne de caractères ou une fonction callable."
            )

        start = time.perf_counter()

        response, usage = with_retry(
            lambda: classify_with_llm(
                question,
                categories,
                additional_prompt=prompt,
                **kwargs
            )
        )

        elapsed = time.perf_counter() - start

        if response is None:
            break

        results.append({
            "Index": idx,
            "Question": question,
            "Réponse": response,
            "Attendue": expected,
            "Correct": response == expected,
            "Temps (s)": elapsed,
            "Total Tokens": usage.total_tokens,
            "Input Tokens": usage.prompt_tokens,
            "Output Tokens": usage.completion_tokens
        })

    return pd.DataFrame(results)

# Test avec 1 ligne de données du dataset

In [19]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "test 1"
model = os.environ["MISTRAL_MODEL"]
results_path = Path("tests", name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index[:1]
    results = test(indices, model=model)
    save_results(results, results_path)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)
print("------------------------")
print_tokens_cost(model, results)

global_metrics.append(metrics)

load_results tests\test 1.pkl

------------------------
test 1
------------------------
Name : test 1
Method : mistral-small-latest
Samples : 1
Accuracy : 100.00%
Precision (macro) : 100.00%
Recall (macro) : 100.00%
F1 (macro) : 100.00%
Precision (weighted) : 100.00%
Recall (weighted) : 100.00%
F1 (weighted) : 100.00%
Temps moyen (s) : 0.541 s
Temps médian (s) : 0.541 s
Temps P95 (s) : 0.541 s
------------------------
                                                                              precision    recall  f1-score   support

Credit reporting, credit repair services, or other personal consumer reports       1.00      1.00      1.00         1

                                                                    accuracy                           1.00         1
                                                                   macro avg       1.00      1.00      1.00         1
                                                                weighted avg       1.00      1.00      1

# Test avec 20 lignes de données du dataset

In [20]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "test 20"
model = os.environ["MISTRAL_MODEL"]
results_path = Path("tests", name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index[:20]
    results = test(indices, model=model)
    save_results(results, results_path)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)
print("------------------------")
print_tokens_cost(model, results)

global_metrics.append(metrics)

load_results tests\test 20.pkl

------------------------
test 20
------------------------
Name : test 20
Method : mistral-small-latest
Samples : 20
Accuracy : 50.00%
Precision (macro) : 21.79%
Recall (macro) : 22.50%
F1 (macro) : 22.12%
Precision (weighted) : 47.86%
Recall (weighted) : 50.00%
F1 (weighted) : 48.85%
Temps moyen (s) : 0.608 s
Temps médian (s) : 0.508 s
Temps P95 (s) : 1.000 s
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.00      0.00      0.00         2
                                                 Credit card or prepaid card       1.00      1.00      1.00         1
Credit reporting, credit repair services, or other personal consumer reports       0.43      0.50      0.46         6
                                                             Debt collection       0.75      0.75      0.75 

## Test avec un prompt plus détaillé

In [21]:

# ---------------------------------------------------------
# Préparation du prompt système
# ---------------------------------------------------------

additional_prompt = """
Pour effectuer la classification :
1. Analysez le problème principal décrit dans la réclamation.
2. Identifiez le produit ou service financier concerné.
3. Comparez le problème avec les définitions des catégories disponibles.
4. Sélectionnez la catégorie qui correspond le mieux au problème principal.
5. Ne sélectionnez jamais une catégorie uniquement parce qu'un mot de la réclamation lui est associé.
"""


In [22]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "test 20 avec prompt plus détaillé"
model = os.environ["MISTRAL_MODEL"]
results_path = Path("tests", name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index[:20]
    results = test(indices, additional_prompt, model=model)
    save_results(results, results_path)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)
print("------------------------")
print_tokens_cost(model, results)

global_metrics.append(metrics)

load_results tests\test 20 avec prompt plus détaillé.pkl
Démarre le test: test 20 avec prompt plus détaillé

------------------------
test 20 avec prompt plus détaillé
------------------------
Name : test 20 avec prompt plus détaillé
Method : mistral-small-latest
Samples : 20
Accuracy : 40.00%
Precision (macro) : 15.48%
Recall (macro) : 19.58%
F1 (macro) : 16.67%
Precision (weighted) : 41.07%
Recall (weighted) : 40.00%
F1 (weighted) : 40.00%
Temps moyen (s) : 0.667 s
Temps médian (s) : 0.489 s
Temps P95 (s) : 1.103 s
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.00      0.00      0.00         2
                                                 Credit card or prepaid card       0.50      1.00      0.67         1
Credit reporting, credit repair services, or other personal consumer reports       0.33      0.33

## Test avec des exemples pour chaque catégorie

In [23]:
examples = (
    df.dropna(subset=["Consumer Claim"])
      .groupby("Tag", group_keys=False)
      .sample(n=3, random_state=42)
      .sort_values("Tag")
)

examples_text = "\n\n".join(
    f"Catégorie : {row['Tag']}\n"
    f"Réclamation : {row['Consumer Claim']}"
    for _, row in examples.iterrows()
)

In [24]:

# ---------------------------------------------------------
# Préparation du prompt système
# ---------------------------------------------------------

additional_prompt = """
Voici des exemples de réclamation correctement classées:
{examples_text}
"""


In [25]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "test 20 avec exemples"
model = os.environ["MISTRAL_MODEL"]
results_path = Path("tests", name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index[:20]
    results = test(indices, additional_prompt, model=model)
    save_results(results, results_path)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)
print("------------------------")
print_tokens_cost(model, results)

global_metrics.append(metrics)

load_results tests\test 20 avec exemples.pkl
Démarre le test: test 20 avec exemples

------------------------
test 20 avec exemples
------------------------
Name : test 20 avec exemples
Method : mistral-small-latest
Samples : 20
Accuracy : 55.00%
Precision (macro) : 32.50%
Recall (macro) : 27.50%
F1 (macro) : 29.17%
Precision (weighted) : 60.00%
Recall (weighted) : 55.00%
F1 (weighted) : 56.67%
Temps moyen (s) : 0.532 s
Temps médian (s) : 0.508 s
Temps P95 (s) : 0.704 s
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       1.00      0.50      0.67         2
                                                 Credit card or prepaid card       1.00      1.00      1.00         1
Credit reporting, credit repair services, or other personal consumer reports       0.50      0.50      0.50         6
                           

## Test avec des exemples du RAG

* Initialise une base de données vectoriel avec le jeu d'entrainement
* Récupère n exemples du système RAG pour insérer au prompt
* Test le LLM

In [26]:
from rag import (
    load_database,
    save_database,
    make_database,
    retrieve_examples,
)

try:
    metadata, index = load_database()
except Exception as e:
    print(e)
    # ----------------------------------------------------------------
    # Construire l'index FAISS uniquement avec le jeu d'entraînement
    # ----------------------------------------------------------------

    X_train  = pd.read_parquet("X_train.parquet")
    y_train  = pd.read_parquet("y_train.parquet")

    # On conserve uniquement les réclamations exploitables
    filter = (X_train["Consumer Claim"].isna() == False)
    X_train  = X_train[filter]
    y_train  = y_train[filter]

    train_data = X_train.copy()

    train_data["Tag"] = y_train["Tag"]

    # Suppression des doublons
    train_data = train_data.drop_duplicates(
        subset=["Consumer Claim"]
    ).reset_index(drop=True)

    print(f"Création de la base de données sur un jeu de {len(train_data)} lignes")

    # ----------------------------------------------------------------
    # Création de la base de données
    # ----------------------------------------------------------------

    metadata, index = make_database(train_data)


    # ----------------------------------------------------------------
    # Sauvegarde de la base de données
    # ----------------------------------------------------------------

    save_database(metadata, index)

index_filename E:\FormationOpenClassRoom\ZenAssist\database\sentence-transformers\all-MiniLM-L6-v2\faiss_index.idx
meta_filename E:\FormationOpenClassRoom\ZenAssist\database\sentence-transformers\all-MiniLM-L6-v2\faiss_index.meta


In [27]:
# ----------------------------------------------------------------
# Crée des exemples
# ----------------------------------------------------------------

def create_samples(question):
    examples = retrieve_examples(
        metadata,
        index,
        question,
        k=10
    )

    examples_text = "\n\n".join(
        f"Catégorie : {row['Tag']}\n"
        f"Réclamation : {row['Consumer Claim']}"
        for _, row in examples.iterrows()
    )

    return f"""
    Voici des exemples de réclamation correctement classées:
    {examples_text}
    """

In [28]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

# ----------------------------------------------------------------
# Test
# ----------------------------------------------------------------

name = "test 20 avec exemples ciblés"
model = os.environ["MISTRAL_MODEL"]
results_path = Path("tests", name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    
    indices = X_test.index[:20]
    results = test(indices, create_samples, model=model)
    save_results(results, results_path)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)
print("------------------------")
print_tokens_cost(model, results)

global_metrics.append(metrics)

load_results tests\test 20 avec exemples ciblés.pkl
Démarre le test: test 20 avec exemples ciblés
Chargement du modèle...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Erreur de l'appel : API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Nouvelle tentative dans 2.12 secondes...
Erreur de l'appel : API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Nouvelle tentative dans 4.01 secondes...
Erreur de l'appel : API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Nouvelle tentative dans 8.65 secondes...
Erreur de l'appel : API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Nouvelle tentative dans 2.31 secondes...
Erreur de l'appel : API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"

In [29]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

# ----------------------------------------------------------------
# Test
# ----------------------------------------------------------------

name = "test 20 avec exemples ciblés et modèle large"
model = os.environ["MISTRAL_LARGE_MODEL"]
results_path = Path("tests", name + ".pkl")
results = load_results(results_path)

if len(results) == 0:
    print("Démarre le test:", name)
    
    indices = X_test.index[:20]
    results = test(indices, create_samples, model=model)
    save_results(results, results_path)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)
print("------------------------")
print_tokens_cost(model, results)

global_metrics.append(metrics)

load_results tests\test 20 avec exemples ciblés et modèle medium.pkl
Démarre le test: test 20 avec exemples ciblés et modèle medium
Erreur de l'appel : API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Nouvelle tentative dans 2.93 secondes...
Erreur de l'appel : API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Nouvelle tentative dans 4.87 secondes...
Erreur de l'appel : API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Nouvelle tentative dans 8.61 secondes...
Erreur de l'appel : API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Nouvelle tentative dans 1

# Evaluation

Evalue le système sur un échantillonnage plus important et affiche les metriques
(le test peut être exécuté en plusieurs fois)

In [30]:

# Sélectionne un nombre equitable d'éléments dans chaque catégorie (évite d'ignorer catégories peut représenté)
max_results = 1000
n_per_category = max_results // len(categories)

samples = (
    y_test.groupby("Tag", group_keys=False)
    .apply(lambda x: x.sample(
        n=min(len(x), n_per_category),
        random_state=42
    ))
).index

print(y_test.loc[samples]["Tag"].value_counts())

Tag
Checking or savings account                                                     100
Credit card or prepaid card                                                     100
Credit reporting, credit repair services, or other personal consumer reports    100
Debt collection                                                                 100
Money transfer, virtual currency, or money service                              100
Mortgage                                                                        100
Payday loan, title loan, or personal loan                                       100
Student loan                                                                    100
Vehicle loan or lease                                                           100
Other financial service                                                          55
Name: count, dtype: int64


In [31]:
name = "test 1000 avec exemples ciblés"
results_path = Path("tests", name + ".pkl")
results = load_results(results_path)

model = os.environ["MISTRAL_MODEL"]
if len(results) < max_results:
    print("Démarre le test:", name)

    # reprend la où ont s'était arrêté (random_state permet de reproduire l'ordre exacte dans indices)
    indices = samples[len(results):max_results]

    additional_results = test(indices, create_samples, model=model)
    results = pd.concat([results, additional_results], ignore_index=True)
    save_results(results, results_path)


load_results tests\test 1000 avec exemples ciblés.pkl
Démarre le test: test 1000 avec exemples ciblés
Erreur de l'appel : API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Nouvelle tentative dans 2.49 secondes...
Erreur de l'appel : API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Nouvelle tentative dans 4.81 secondes...
Erreur de l'appel : API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Nouvelle tentative dans 8.45 secondes...
Erreur de l'appel : API error occurred: Status 429. Body: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}
Nouvelle tentative dans 16.28 secondes...
Erreur de l'a

In [32]:
results.head(10)

,Index,Question,Réponse,Attendue,Correct,Temps (s),Total Tokens,Input Tokens,Output Tokens
0,450589,I received a phone call today telling me a cou...,Debt collection,Checking or savings account,False,0.48,3924,3920,4
1,673975,On XXXX i closed my Wells Fargo Bank checking ...,Checking or savings account,Checking or savings account,True,0.50,3358,3353,5
2,89281,"on XX/XX/16, I received a notification from my...",Checking or savings account,Checking or savings account,True,0.82,4274,4269,5
3,646579,I had a very unpleasant experience today with ...,Checking or savings account,Checking or savings account,True,0.63,5641,5636,5
4,171628,I went to the companys ATM and withdraws some ...,Checking or savings account,Checking or savings account,True,0.54,1874,1869,5
5,317432,"I asked CitizensBank, through their email syst...",Checking or savings account,Checking or savings account,True,0.71,4673,4668,5
6,430593,Wells Fargo sat in front of my uncle and I and...,Credit card or prepaid card,Checking or savings account,False,0.47,4657,4650,7
7,53882,At the time of account opening {$200.00} bonus...,Checking or savings account,Checking or savings account,True,0.40,1848,1843,5
8,109215,I suffered severe financial damages as a resul...,Checking or savings account,Checking or savings account,True,0.41,5109,5104,5
9,292941,We have made requests from three national bank...,Checking or savings account,Checking or savings account,True,0.35,2837,2832,5


In [33]:
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

metrics = calculate_metrics(name, model, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)
print("------------------------")
print_tokens_cost(model, results)

global_metrics.append(metrics)


------------------------
test 1000 avec exemples ciblés
------------------------
Name : test 1000 avec exemples ciblés
Method : mistral-small-latest
Samples : 955
Accuracy : 71.94%
Precision (macro) : 66.88%
Recall (macro) : 68.70%
F1 (macro) : 66.19%
Precision (weighted) : 70.03%
Recall (weighted) : 71.94%
F1 (weighted) : 69.31%
Temps moyen (s) : 4.229 s
Temps médian (s) : 3.339 s
Temps P95 (s) : 8.546 s
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.73      0.80      0.76       100
                                                 Credit card or prepaid card       0.71      0.90      0.80       100
Credit reporting, credit repair services, or other personal consumer reports       0.52      0.93      0.67       100
                                                             Debt collection       0.74     

# Tableau comparatif synthétique des métriques

In [37]:
global_metrics_df = pd.DataFrame(
    global_metrics,
    columns=["Name", "Samples", "Method", "Accuracy", "F1 (macro)", "Temps moyen (s)"]
    )

global_metrics_df = global_metrics_df.rename(columns={
    "Name" : "Configuration du Test",
    "Samples" : "Échantillon (N)",
    "Method" : "Modèle LLM",
    "Accuracy": "Accuracy",
    "F1 (macro)": "F1-Score",
    "Temps moyen (s)" : "Temps Moyen (s)"
})

pd.set_option("display.float_format", "{:,.3f}".format)

global_metrics_df

,Configuration du Test,Échantillon (N),Modèle LLM,Accuracy,F1-Score,Temps Moyen (s)
0,test 1,1,mistral-small-latest,1.000,1.000,0.541
1,test 20,20,mistral-small-latest,0.500,0.221,0.608
2,test 20 avec prompt plus détaillé,20,mistral-small-latest,0.400,0.167,0.667
3,test 20 avec exemples,20,mistral-small-latest,0.550,0.292,0.532
4,test 20 avec exemples ciblés,20,mistral-small-latest,0.700,0.455,3.496
5,test 20 avec exemples ciblés et modèle medium,20,mistral-medium-latest,0.850,0.634,7.152
6,test 1000 avec exemples ciblés,955,mistral-small-latest,0.719,0.662,4.229


## Analyse par catégorie

L'accuracy globale de 86 % ne permet pas de savoir si le modèle sait correctement traiter les petites catégories.

In [36]:
from sklearn.metrics import classification_report

name = "test 1000 avec exemples ciblés"
results_path = Path("tests", name + ".pkl")
results = load_results(results_path)


y_test = results["Attendue"]
y_pred_svc = results["Réponse"]

print(
    classification_report(
        y_test,
        y_pred_svc,
        zero_division=0
    )
)

load_results tests\test 1000 avec exemples ciblés.pkl
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.73      0.80      0.76       100
                                                 Credit card or prepaid card       0.71      0.90      0.80       100
Credit reporting, credit repair services, or other personal consumer reports       0.52      0.93      0.67       100
                                                             Debt collection       0.74      0.67      0.70       100
                          Money transfer, virtual currency, or money service       0.81      0.77      0.79       100
                                                                    Mortgage       0.88      0.90      0.89       100
                                                     Other financial service       0.00      0.00      0.00        55
 

In [39]:
results[results["Attendue"] == "Other financial service"]

,Index,Question,Réponse,Attendue,Correct,Temps (s),Total Tokens,Input Tokens,Output Tokens
600,604232,Returned check amounts for XXXX are showing as...,"Credit reporting, credit repair services, or o...",Other financial service,False,0.599,3158,3145,13
601,658927,XXXX XXXX XXXX XXXX XXXX XXXX XXXX. XXXX . XXX...,"Credit reporting, credit repair services, or o...",Other financial service,False,7.367,2834,2821,13
602,584979,In XXXX a company I work for in the United Sta...,"Money transfer, virtual currency, or money ser...",Other financial service,False,0.487,4172,4162,10
603,871984,CareCredit/Synchrony Bank has placed an excess...,Credit card or prepaid card,Other financial service,False,9.855,2283,2276,7
604,632186,I cashed a check written on a jp mogan check a...,Checking or savings account,Other financial service,False,0.364,4777,4772,5
605,736482,I purchased a money order pay to the order of ...,"Money transfer, virtual currency, or money ser...",Other financial service,False,3.141,2699,2689,10
606,604257,I am dealing with www.Remitly.com. This websit...,"Money transfer, virtual currency, or money ser...",Other financial service,False,3.368,3187,3177,10
607,762254,I have just started with Clear One Advantage a...,"Payday loan, title loan, or personal loan",Other financial service,False,3.206,2406,2395,11
608,771158,The American Student Services has taken owners...,"Credit reporting, credit repair services, or o...",Other financial service,False,3.219,2394,2381,13
609,605727,"First off, the money is not lost or stolen. It...","Money transfer, virtual currency, or money ser...",Other financial service,False,2.805,4339,4329,10
